# 02 — Line vs. Polygon Basics

A polygon is not one thing — it is a **closed chain of segments**. That means "does this path cross this country?" reduces to a question you already know how to answer: "does this path segment cross any of those edge segments?"

```
line intersects polygon  ⟺  line intersects ANY edge of the polygon
```

This notebook builds that loop, then adds a bounding-box pre-check to skip polygons that cannot possibly intersect.

## Setup

In [1]:
import json
from pathlib import Path
from ipyleaflet import Map, GeoJSON, basemaps

DATA_DIR = Path("data")

with open(DATA_DIR / "paths.geojson") as f:
    module_paths = json.load(f)


# Intersection helpers from notebook 01 — redefined here for self-containment
def orientation(p, q, r):
    return (q[0] - p[0]) * (r[1] - q[1]) - (q[1] - p[1]) * (r[0] - q[0])

def on_segment(p, q, r):
    return (min(p[0], r[0]) <= q[0] <= max(p[0], r[0]) and
            min(p[1], r[1]) <= q[1] <= max(p[1], r[1]))

def segments_intersect(p1, p2, p3, p4):
    d1 = orientation(p3, p4, p1)
    d2 = orientation(p3, p4, p2)
    d3 = orientation(p1, p2, p3)
    d4 = orientation(p1, p2, p4)
    if (d1 > 0 and d2 < 0 or d1 < 0 and d2 > 0) and \
       (d3 > 0 and d4 < 0 or d3 < 0 and d4 > 0):
        return True
    if d1 == 0 and on_segment(p3, p1, p4): return True
    if d2 == 0 and on_segment(p3, p2, p4): return True
    if d3 == 0 and on_segment(p1, p3, p2): return True
    if d4 == 0 and on_segment(p1, p4, p2): return True
    return False

print("Ready.")

Ready.


## A Polygon is a Closed Ring of Segments

In GeoJSON, a `Polygon` geometry holds one or more **rings** — each ring is a list of `[lon, lat]` positions where the first and last position are identical (the ring is closed).

```json
{
  "type": "Polygon",
  "coordinates": [
    [ [lon0,lat0], [lon1,lat1], [lon2,lat2], [lon0,lat0] ]  ← outer ring
  ]
}
```

The first ring is the outer boundary. Any additional rings are **holes** (interior cutouts). For intersection testing, both rings matter — a hole is still a boundary a path can cross.

Each consecutive pair of positions in a ring is one **edge segment**. A ring with `n` positions has `n − 1` edges (the closing edge connects the last non-duplicate position back to the first).

In [2]:
# A simple square polygon — 4 corners, 4 edges
square = {
    "type": "Polygon",
    "coordinates": [[
        [20.0, 40.0],   # top-left
        [30.0, 40.0],   # top-right
        [30.0, 30.0],   # bottom-right
        [20.0, 30.0],   # bottom-left
        [20.0, 40.0],   # closed — same as first
    ]]
}

ring = square["coordinates"][0]
print(f"Ring has {len(ring)} positions → {len(ring) - 1} edges")
for i in range(len(ring) - 1):
    print(f"  Edge {i + 1}: {ring[i]}  →  {ring[i + 1]}")

Ring has 5 positions → 4 edges
  Edge 1: [20.0, 40.0]  →  [30.0, 40.0]
  Edge 2: [30.0, 40.0]  →  [30.0, 30.0]
  Edge 3: [30.0, 30.0]  →  [20.0, 30.0]
  Edge 4: [20.0, 30.0]  →  [20.0, 40.0]


## Extracting Edges

A reusable `polygon_edges` function should handle both `Polygon` and `MultiPolygon` geometries — many countries in real datasets are multi-part (islands, exclaves).

```
Polygon      → coordinates = [ ring, ring, ... ]
MultiPolygon → coordinates = [ [ring, ring], [ring, ring], ... ]
```

In [3]:
def polygon_edges(geometry):
    """
    Return all edges of a Polygon or MultiPolygon geometry
    as a list of (p1, p2) coordinate pairs.
    """
    edges = []

    if geometry["type"] == "Polygon":
        rings = geometry["coordinates"]
    elif geometry["type"] == "MultiPolygon":
        # flatten: each polygon has a list of rings
        rings = [ring for poly in geometry["coordinates"] for ring in poly]
    else:
        return edges

    for ring in rings:
        for i in range(len(ring) - 1):
            edges.append((ring[i], ring[i + 1]))

    return edges


edges = polygon_edges(square)
print(f"{len(edges)} edges extracted from square")
for e in edges:
    print(f"  {e[0]}  →  {e[1]}")

4 edges extracted from square
  [20.0, 40.0]  →  [30.0, 40.0]
  [30.0, 40.0]  →  [30.0, 30.0]
  [30.0, 30.0]  →  [20.0, 30.0]
  [20.0, 30.0]  →  [20.0, 40.0]


## Line Intersects Polygon ⟺ Line Intersects Any Edge

To test whether a line segment crosses a polygon, iterate over every edge and call `segments_intersect`. Stop as soon as one edge returns `True` — no need to check the rest.

```
for each edge in polygon:
    if segments_intersect(path_start, path_end, edge_start, edge_end):
        return True
return False
```

This is an **O(n)** test where `n` is the number of polygon edges — for complex country borders, that can be thousands.

In [5]:
def line_crosses_polygon(p1, p2, geometry):
    """True if segment p1-p2 intersects any edge of the geometry."""
    for e1, e2 in polygon_edges(geometry):
        if segments_intersect(p1, p2, e1, e2):
            return True
    return False


# Line that clearly crosses the square (diagonal)
print("Diagonal cross:  ", line_crosses_polygon([15, 35], [35, 35], square))  # True

# Line entirely outside the square
print("Outside:         ", line_crosses_polygon([0, 35], [10, 35], square))   # False

# Line entirely inside the square — no edge crossed
print("Fully inside:    ", line_crosses_polygon([22, 38], [28, 32], square))  # False

Diagonal cross:   True
Outside:          False
Fully inside:     False


### The Fully-Inside Case

Notice that a segment entirely **inside** a polygon returns `False` — it does not cross any edge. For most use cases (missile paths, routes), this is correct: a path that starts and ends inside a country has already been placed there; we care about entry and exit crossings.

If you need to detect containment as well, that requires a separate **point-in-polygon** test — a topic for a later module.

## Bounding Box Pre-Check

Testing thousands of edges for every path-polygon pair is expensive. A **bounding box check** provides a fast early-exit: if the path's axis-aligned bounding box does not overlap the polygon's bounding box, the path cannot possibly cross the polygon — skip it entirely.

```
bboxes overlap?  →  No  →  skip polygon (fast)
                 →  Yes →  run full edge test
```

This is an **O(1)** reject that avoids the O(n) edge loop for the majority of polygons on a world map.

In [6]:
def segment_bbox(p1, p2):
    """Return (min_lon, min_lat, max_lon, max_lat) for a segment."""
    return (min(p1[0], p2[0]), min(p1[1], p2[1]),
            max(p1[0], p2[0]), max(p1[1], p2[1]))


def geometry_bbox(geometry):
    """Return (min_lon, min_lat, max_lon, max_lat) for any polygon geometry."""
    if geometry["type"] == "Polygon":
        rings = geometry["coordinates"]
    else:
        rings = [ring for poly in geometry["coordinates"] for ring in poly]

    all_coords = [pt for ring in rings for pt in ring]
    lons = [c[0] for c in all_coords]
    lats = [c[1] for c in all_coords]
    return (min(lons), min(lats), max(lons), max(lats))


def bboxes_overlap(b1, b2):
    """True if two (min_lon, min_lat, max_lon, max_lat) boxes share any area."""
    return not (b1[2] < b2[0] or b2[2] < b1[0] or   # one is left of the other
                b1[3] < b2[1] or b2[3] < b1[1])       # one is above the other


def line_crosses_polygon_fast(p1, p2, geometry):
    """Bbox-accelerated version: skips edge test when boxes don't overlap."""
    if not bboxes_overlap(segment_bbox(p1, p2), geometry_bbox(geometry)):
        return False
    return line_crosses_polygon(p1, p2, geometry)


# Same tests — results identical, but the outside case exits before any edge test
print("Diagonal cross (fast): ", line_crosses_polygon_fast([15, 35], [35, 35], square))
print("Outside       (fast): ", line_crosses_polygon_fast([0, 35], [10, 35], square))

Diagonal cross (fast):  True
Outside       (fast):  False


## Exercise A

The polygon below is a rough bounding outline of Iraq.

```python
iraq_rough = {
    "type": "Polygon",
    "coordinates": [[
        [38.8, 37.4], [48.8, 37.4], [48.8, 29.1],
        [44.7, 29.1], [38.8, 33.0], [38.8, 37.4]
    ]]
}
```

1. Use `polygon_edges` to extract and print all edges.
2. Count the edges and confirm it equals the number of ring positions minus one.
3. Visualize the polygon on a map centered over the Middle East.

In [ ]:
from ipyleaflet import Map, GeoJSON, basemaps

iraq_rough = {
    "type": "Polygon",
    "coordinates": [[
        [38.8, 37.4], [48.8, 37.4], [48.8, 29.1],
        [44.7, 29.1], [38.8, 33.0], [38.8, 37.4]
    ]]
}

# 1. Extract and print all edges
# 2. Count edges vs ring positions
# 3. Display on a map
# Your code here

## Exercise B

The **Alpha** path from `paths.geojson` runs from Washington D.C. to Tehran. Test it against `iraq_rough` from Exercise A.

1. Extract the Alpha path segment (start and end coordinates).
2. Call `line_crosses_polygon` and print the result.
3. Call `line_crosses_polygon_fast` and confirm it returns the same result.
4. Display both the path and the polygon on the same map to visually verify.

In [ ]:
from ipyleaflet import Map, GeoJSON, basemaps

# 1. Get Alpha path start/end coordinates from module_paths
# 2. Test with line_crosses_polygon
# 3. Test with line_crosses_polygon_fast and compare
# 4. Display both on a map
# Your code here

---

## Check Your Understanding

**1.** Why is a polygon equivalent to a collection of line segments for the purpose of intersection testing?

**2.** An N-vertex polygon ring in GeoJSON has how many edges? Why is it not N?

```python
# No code needed — answer in your own words
```

## Next

In [03 — Detecting Intersections](./03-Detecting_Intersections.ipynb), we scale up to a full country dataset — looping over every polygon, applying the fast test, and collecting which features each path actually crosses.